<a href="https://colab.research.google.com/github/kumarsusheel7497-collab/CODE_ALPHA-1/blob/main/CodeAlpha_EmotionRecognitionFromSpeech.ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.utils import to_categorical

dataset_path = "dataset"

X = []
y = []

emotion_dict = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

# Ensure the dataset directory exists
if not os.path.exists(dataset_path):
    print(f"Error: Dataset path '{dataset_path}' not found. Please ensure the directory exists and contains audio files.")
else:
    # Initialize an empty list to store all file names found
    all_files_in_dataset = []
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if file.endswith(".wav"):
                all_files_in_dataset.append(file)
                file_path = os.path.join(root, file)

                try:
                    # Extract emotion code from filename (assuming RAVDESS-like format)
                    emotion_code = file.split("-")[2]
                    emotion = emotion_dict[emotion_code]

                    # Load audio file and extract MFCC features
                    audio, sr = librosa.load(file_path, duration=3, offset=0.5)

                    # Ensure mfcc has the same number of features as expected by the model (40)
                    mfcc = np.mean(
                        librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40).T,
                        axis=0
                    )

                    X.append(mfcc)
                    y.append(emotion)
                except Exception as e:
                    print(f"Could not process file {file_path}: {e}")

X = np.array(X)

# Check if any data was loaded before proceeding
if X.size == 0:
    print("No audio files were processed or found in the dataset. Please check the 'dataset' directory and file format.")
elif X.shape[0] < 2: # Check if there's enough data to split
    print(f"Only {X.shape[0]} audio file(s) found. Need at least 2 files to perform a train-test split and train the model. Please upload more files to the 'dataset' directory.")
else:
    encoder = LabelEncoder()
    y = encoder.fit_transform(y)
    y = to_categorical(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = Sequential([
        Input(shape=(40,)), # Explicitly define the input shape
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(y.shape[1], activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    print("\nStarting model training...")
    model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)
    print("Model training complete.")

    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

    print(f"\nAccuracy: {accuracy*100:.2f}%")

    # --- Prediction part ---
    # Define a sample audio path for prediction. You can change this to any .wav file.
    # For demonstration, we'll use the first file processed from the dataset if available.
    sample_audio_path = None
    if len(all_files_in_dataset) > 0: # Use the list of all files found
        # Just pick the first file for demonstration
        sample_audio_path = os.path.join(dataset_path, all_files_in_dataset[0])

    if sample_audio_path and os.path.exists(sample_audio_path):
        try:
            print(f"\nAttempting prediction for: {sample_audio_path}")
            sample_audio_raw, sample_sr = librosa.load(sample_audio_path, duration=3, offset=0.5)
            sample_audio_mfcc = np.mean(librosa.feature.mfcc(y=sample_audio_raw, sr=sample_sr, n_mfcc=40).T, axis=0)

            # Reshape the MFCC features to match the model's input (1 sample, 40 features)
            sample_audio = np.array([sample_audio_mfcc])

            prediction = model.predict(sample_audio)
            emotion = encoder.inverse_transform([np.argmax(prediction)])
            print("Predicted Emotion:", emotion[0])
        except Exception as e:
            print(f"Could not perform prediction for {sample_audio_path}: {e}")
    else:
        print("\nNo sample audio file available for prediction. Ensure 'dataset' has files and sample_audio_path is set correctly.")


Starting model training...
Model training complete.

Accuracy: 60.00%

Attempting prediction for: dataset/03-01-02-02-01-01-01.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
Predicted Emotion: calm


### 1. Create the `dataset` directory

First, we need to ensure the `dataset` directory exists. This cell will create it if it doesn't already.

In [ ]:
import os

dataset_path = "dataset"

if not os.path.exists(dataset_path):
    os.makedirs(dataset_path)
    print(f"Directory '{dataset_path}' created.")
else:
    print(f"Directory '{dataset_path}' already exists.")

Directory 'dataset' already exists.


### 2. Uploading Files to the `dataset` Directory

You have two main options to upload your `.wav` files into the newly created `dataset` folder:

#### Option A: Upload Directly from Your Computer

This method is good for a few files. It will open a file selection dialog. Make sure to select all the `.wav` files you want to upload.

In [ ]:
from google.colab import files
import shutil

print(f"Uploading files to '{dataset_path}'...")

uploaded = files.upload()

for filename in uploaded.keys():
    destination_path = os.path.join(dataset_path, filename)
    with open(destination_path, 'wb') as f:
        f.write(uploaded[filename])
    print(f"File '{filename}' uploaded to '{destination_path}'")

Uploading files to 'dataset'...


Saving 03-01-01-01-01-01-01.wav to 03-01-01-01-01-01-01 (2).wav
Saving 03-01-01-01-01-02-01.wav to 03-01-01-01-01-02-01 (1).wav
Saving 03-01-01-01-02-01-01.wav to 03-01-01-01-02-01-01 (1).wav
Saving 03-01-01-01-02-02-01.wav to 03-01-01-01-02-02-01 (1).wav
Saving 03-01-02-01-01-01-01.wav to 03-01-02-01-01-01-01 (1).wav
Saving 03-01-02-01-01-02-01.wav to 03-01-02-01-01-02-01 (1).wav
Saving 03-01-02-01-02-01-01.wav to 03-01-02-01-02-01-01 (1).wav
Saving 03-01-02-01-02-02-01.wav to 03-01-02-01-02-02-01 (1).wav
Saving 03-01-02-02-01-01-01.wav to 03-01-02-02-01-01-01.wav
Saving 03-01-02-02-01-02-01.wav to 03-01-02-02-01-02-01.wav
Saving 03-01-02-02-02-01-01.wav to 03-01-02-02-02-01-01.wav
Saving 03-01-02-02-02-02-01.wav to 03-01-02-02-02-02-01.wav
Saving 03-01-03-01-01-01-01.wav to 03-01-03-01-01-01-01.wav
Saving 03-01-03-01-01-02-01.wav to 03-01-03-01-01-02-01.wav
Saving 03-01-03-01-02-01-01.wav to 03-01-03-01-02-01-01.wav
Saving 03-01-03-01-02-02-01.wav to 03-01-03-01-02-02-01.wav
File '03

#### Option B: Mount Google Drive

If your `.wav` files are already in your Google Drive, mounting your Drive is often the most convenient option, especially for larger datasets or if you want to keep your data organized in Drive.

After running the cell below, you'll be prompted to authorize Colab to access your Google Drive. Follow the instructions to get an authorization code.

Once your Drive is mounted, you can then copy files from your Drive to the `dataset` folder. Replace `/content/gdrive/My Drive/Your_Audio_Files_Folder/` with the actual path to your `.wav` files in Google Drive.

**Example:** If your audio files are in a folder named `MyAudioData` inside 'My Drive' on Google Drive, and you want to copy all `.wav` files from there to the `dataset` folder, you would use something like this:

In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')
print("Google Drive mounted at /content/gdrive")


Mounted at /content/gdrive
Google Drive mounted at /content/gdrive


In [ ]:
import shutil
import glob

source_dir = '/content/gdrive/My Drive/MyAudioData/' # <--- **UPDATE THIS PATH**

if os.path.exists(source_dir):
    for audio_file in glob.glob(os.path.join(source_dir, '*.wav')):
        shutil.copy(audio_file, dataset_path)
        print(f"Copied {os.path.basename(audio_file)} to {dataset_path}")
else:
    print(f"Error: Source directory '{source_dir}' not found in Google Drive. Please check the path.")

Error: Source directory '/content/gdrive/My Drive/MyAudioData/' not found in Google Drive. Please check the path.


After uploading your files using either Option A or Option B, you can then re-run the original cell (`7RyGAfA_ZORc`) to process the data.